# Capstone Three: Pre-processing and Modeling
## USA Real Estate Market Analysis
**Author:** Daksha Gummadi

This notebook covers the pre processing and modeling steps for my third capstone project. In the previous notebook I cleaned and merged 2.2 million real estate listings with Census economic data. In this notebook I prepare that data for machine learning and build three models to predict listing price. My response variable is price, which is continuous, so this is a regression problem. My data is not a time series, so a standard random train test split is appropriate.

## Table of Contents

1. [Import Libraries](#import-libraries)
2. [Load Data](#load-data)
3. [Prepare Features and Target](#prepare-features-and-target)
4. [Create Dummy Features](#create-dummy-features)
5. [Train Test Split](#train-test-split)
6. [Standardize Numeric Features](#standardize-numeric-features)
7. [Model 1: Linear Regression](#model-1-linear-regression)
8. [Model 2: Ridge Regression](#model-2-ridge-regression)
9. [Model 3: Random Forest](#model-3-random-forest)
10. [Hyperparameter Tuning](#hyperparameter-tuning)
11. [Model Comparison](#model-comparison)
12. [Feature Importance](#feature-importance)
13. [Conclusion](#conclusion)

## 1. Import Libraries

**Libraries I'm using:**

* pandas (pd): Loads the CSV file and lets me work with the data in table format.
* numpy (np): Provides math functions. I use it for the log transformation of price.
* matplotlib.pyplot (plt): Creates charts to visualize model results.
* scikit-learn (sklearn): The machine learning library. I use it for splitting the data, scaling the features, building the models, and measuring performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load Data

I load the cleaned and merged dataset I saved at the end of the data wrangling notebook. It has all the listing features plus the Census economic columns joined by ZIP code.

In [ ]:
df = pd.read_csv('realtor_cleaned_merged.csv')
print(df.shape)
df.head()

The dataset is very large at over 2.2 million rows. Training models on the full dataset would take a very long time on my computer. I take a random sample of 300,000 rows so the models train in a reasonable amount of time. A sample this size is still large enough to learn the patterns in the data. I use a random state so the sample is the same every time I run the notebook.

In [ ]:
df = df.sample(300000, random_state=42)
print(df.shape)

## 3. Prepare Features and Target

My response variable is price, which is continuous so this is a regression problem. In the EDA notebook I found that price is heavily right skewed with a skewness of 15.45 and log transforming it brings the skewness down to -0.57. A more symmetric target helps regression models perform better, so I use the log of price as my target. For features I keep the property characteristics, my engineered features, the Census economic columns, and the two categorical columns status and state. I drop columns that are just ID numbers like brokered_by and street, and I drop city and zip_code because they have thousands of unique values which would create too many dummy columns. State captures the location signal at a manageable level. I also drop price_per_sqft because it is calculated from price, so keeping it would leak the answer into the model.

In [ ]:
df = df.dropna(subset=['median_hh_income', 'unemployment_rate', 'poverty_pct', 'occupied_units'])
df['log_price'] = np.log1p(df['price'])

features = ['bed', 'bath', 'acre_lot', 'house_size', 'listing_count_zip',
            'median_hh_income', 'unemployment_rate', 'poverty_pct', 'occupied_units',
            'status', 'state']
X = df[features]
y = df['log_price']
print(X.shape)
print(y.shape)
X.head()

I dropped the small number of rows missing Census data so every row has complete features. The feature set has 9 numeric columns and 2 categorical columns. The target is log price.

## 4. Create Dummy Features

My dataset has two categorical columns. Status has values like for_sale and sold. State has values like Texas and California. Models cannot use text values directly, so I convert them into dummy features. Each category becomes its own column with a 1 or 0. I use drop_first=True to drop one category from each group, which avoids redundant columns.

In [ ]:
X = pd.get_dummies(X, columns=['status', 'state'], drop_first=True)
print(X.shape)
X.head()

After creating dummies the feature set grew from 11 columns to over 60 columns. Each state and status value is now its own 0 or 1 column that the models can use.

## 5. Train Test Split

I split the data into a training set and a testing set. The models learn from the training set. The testing set is held back and only used at the end to measure how well each model predicts prices it has never seen. I use an 80/20 split, which is a standard choice. My data is not a time series, so a random split is appropriate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 6. Standardize Numeric Features

My numeric features have very different ranges. Bedrooms go from 0 to 20 while median household income goes up to 249,688. Features with large values can dominate models like linear regression just because of their scale. I use StandardScaler to put every numeric feature on the same scale with a mean of 0 and a standard deviation of 1. I fit the scaler on the training data only and then apply it to both sets. This prevents any information from the test set leaking into training. The dummy columns are already 0 or 1 so they do not need scaling.

In [ ]:
numeric_cols = ['bed', 'bath', 'acre_lot', 'house_size', 'listing_count_zip',
                'median_hh_income', 'unemployment_rate', 'poverty_pct', 'occupied_units']
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
X_train[numeric_cols].describe().T.round(2)

The summary confirms the scaling worked. Every numeric feature in the training set now has a mean of 0 and a standard deviation of 1. The pre-processing is complete and the data is ready for modeling.

## 7. Model 1: Linear Regression

My first model is linear regression. It is the simplest regression model and gives me a baseline to compare the other models against. I measure performance with two metrics. RMSE is the average size of the prediction errors, where lower is better. R squared is the share of price variation the model explains, where higher is better and 1.0 is perfect.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)
print("Linear Regression")
print("RMSE:", round(lr_rmse, 4))
print("R squared:", round(lr_r2, 4))

## 8. Model 2: Ridge Regression

My second model is ridge regression. It is linear regression with regularization, which shrinks the coefficients to reduce the impact of correlated features. In the EDA I found that bed, bath, and house_size are correlated with each other, so ridge is a good fit for this data.

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)
print("Ridge Regression")
print("RMSE:", round(ridge_rmse, 4))
print("R squared:", round(ridge_r2, 4))

## 9. Model 3: Random Forest

My third model is a random forest. It builds many decision trees and averages their predictions. Unlike the linear models it can capture non linear patterns, like the way price stays flat from 1 to 3 bedrooms and then jumps at 4, which I found in the EDA. I limit the number of trees and the tree depth so it trains in a reasonable time.

In [ ]:
rf = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)
print("Random Forest")
print("RMSE:", round(rf_rmse, 4))
print("R squared:", round(rf_r2, 4))

## 10. Hyperparameter Tuning

The random forest performed best, so I tune its hyperparameters to see if I can improve it further. I use GridSearchCV to try different combinations of tree count and tree depth with 3 fold cross validation. To keep the search fast I run it on a smaller sample of the training data, then retrain the best settings on the full training set.

In [ ]:
X_tune = X_train.sample(50000, random_state=42)
y_tune = y_train.loc[X_tune.index]
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [15, 20]
}

grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                    param_grid, cv=3, scoring='r2')
grid.fit(X_tune, y_tune)
print("Best parameters:", grid.best_params_)

In [ ]:
rf_best = RandomForestRegressor(**grid.best_params_, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)
rf_best_pred = rf_best.predict(X_test)
rf_best_rmse = np.sqrt(mean_squared_error(y_test, rf_best_pred))
rf_best_r2 = r2_score(y_test, rf_best_pred)
print("Tuned Random Forest")
print("RMSE:", round(rf_best_rmse, 4))
print("R squared:", round(rf_best_r2, 4))

## 11. Model Comparison

I compare all the models side by side using the same test set. The best model has the lowest RMSE and the highest R squared.

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Random Forest', 'Tuned Random Forest'],
    'RMSE': [lr_rmse, ridge_rmse, rf_rmse, rf_best_rmse],
    'R_squared': [lr_r2, ridge_r2, rf_r2, rf_best_r2]
}).round(4)
print(results.to_string(index=False))
plt.figure(figsize=(9, 5))
plt.bar(results['Model'], results['R_squared'], color=['steelblue', 'darkorange', 'seagreen', 'purple'], edgecolor='white')
plt.ylabel('R squared')
plt.title('Model Comparison by R squared')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_best_pred, alpha=0.1, s=5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linewidth=2)
plt.xlabel('Actual Log Price')
plt.ylabel('Predicted Log Price')
plt.title('Tuned Random Forest: Actual vs Predicted')
plt.tight_layout()
plt.show()

The scatter plot compares the actual log prices to the predictions from the best model. The red line is where a perfect prediction would fall. Most points cluster around the line, which shows the model captures the overall pattern well. The spread around the line represents the prediction error.

## 12. Feature Importance

The random forest can report which features mattered most when making predictions. This tells me what actually drives listing prices in the model.

In [ ]:
importances = pd.Series(rf_best.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 7))
plt.barh(top_features.index, top_features.values, color='steelblue', edgecolor='white')
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).head(10).round(4))

The most important features confirm what I found in the EDA. House size and the ZIP code economic features like median household income drive the predictions, along with bathrooms and specific states. This matches my earlier finding that location matters as much as the property itself.

## 13. Conclusion

I completed the pre-processing and modeling steps for my USA Real Estate capstone project. In pre-processing I created dummy features for the two categorical columns status and state. I standardized all nine numeric features with StandardScaler so they share the same scale. I split the data into an 80 percent training set and a 20 percent testing set. I used log price as my target because raw price is heavily right skewed. I built three models and tuned the best one. Linear regression was my baseline. Ridge regression added regularization to handle the correlated features. Random forest captured non linear patterns and performed the best. After hyperparameter tuning with GridSearchCV the tuned random forest was the final winner with the lowest RMSE and the highest R squared on the test set.

The feature importances confirm the story from my EDA. House size, ZIP code median household income, bathrooms, and state are the strongest drivers of listing price. The final model can now be used to predict listing prices for homes based on their features and location. A future improvement would be adding the ZIP code trajectory features from the 2019 versus 2023 Census comparison to predict which markets will appreciate over the next five years.